# 02 — Gradients and the learning rate

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/omarahmedZa/ai-ml-course/blob/main/notebooks/002-slope-to-gradient.ipynb)

Training is a walk downhill: measure the slope, take a small step against it, repeat. In this notebook you take three steps by hand on f(w) = w squared, then watch what happens when the step size is too small, just right, and too large. All code runs on a fresh Colab runtime.

## 1. The numbers

The loss f(w) = w² is a bowl with its bottom at w = 0. The slope at any point is 2w, so at w = 2 the slope is 4 and the loss is 4. Notebook 004 reported a loss of 9 from y = 10 and ŷ = 7; the arithmetic here is the same shape, just smaller.

In [ ]:
import numpy as np                             # NumPy is preinstalled on Colab
w = 2.0                                       # our starting weight, the same w as notebook 004
eta = 0.1                                     # the learning rate: how far we move per step
loss = w ** 2                                 # f(w) = w squared, currently 4.0
grad = 2 * w                                  # the derivative of w squared is 2w
next_w = w - eta * grad                       # descend: subtract eta times the slope
print("loss f(w) =", loss)                    # 4.0
print("gradient =", grad)                     # 4.0
print("next w =", next_w)                     # 1.6
print("004 check: (10 - 7)^2 =", (10.0 - 7.0) ** 2)   # 9.0, the loss from notebook 004

**Line by line**

- `import numpy as np` — loads NumPy, the array library Colab already has.
- `w = 2.0` — the starting weight; `f` is a function of this one number.
- `eta = 0.1` — the learning rate, written η, controlling the step length.
- `loss = w ** 2` — the value of f at the current weight.
- `grad = 2 * w` — the gradient: the slope of f at w, equal to 4 here.
- `next_w = w - eta * grad` — move against the slope by `eta`; 2 − 0.4 = 1.6.
- `print("loss f(w) =" ...)` — shows the starting value, 4.0.
- `print("gradient =" ...)` — shows the slope, 4.0.
- `print("next w =" ...)` — shows where one step lands, 1.6.
- `print("004 check" ...)` — recomputes the 9 from notebook 004 for continuity.

## 2. Step by step

One step is not training. Repeat the slope-then-step recipe three times and watch the weight and the loss shrink together. The loss falls by less each time because the bowl gets flatter near the bottom.

In [ ]:
w = 2.0                                       # reset the weight to the start
eta = 0.1                                     # the same learning rate as section 1
history = [w]                                 # keep every weight we visit

for step in range(3):                         # take exactly three descent steps
    grad = 2 * w                              # slope of f at the current weight
    w = w - eta * grad                        # the update rule, one line
    history.append(w)                         # record the new weight
    print(f"step {step + 1}: w = {w:.4f}  loss = {w ** 2:.4f}")   # print the table row

**Line by line**

- `w = 2.0` — start from the same weight as section 1.
- `eta = 0.1` — keep the step size fixed while we iterate.
- `history = [w]` — a list that starts with the initial weight.
- `for step in range(3)` — three iterations, numbered 0, 1, 2 inside the loop.
- `grad = 2 * w` — recompute the slope at the new weight each time.
- `w = w - eta * grad` — the same update as before, now in a loop.
- `history.append(w)` — store the weight so later cells can plot the path.
- `print(f"step ...")` — one row per step: weight and loss to four decimals.

## 3. Plot: loss versus step for three learning rates

The learning rate decides the whole story. A small η creeps downhill, a moderate η arrives quickly, and η = 1.0 jumps from +2 to −2 and back, so the loss never changes.

In [ ]:
import matplotlib.pyplot as plt                # plotting library, preinstalled on Colab
import numpy as np                             # arrays for the curves

def losses(eta, steps=12, start=2.0):         # loss curve for one learning rate
    w = start                                  # every run starts at the same weight
    curve = [w ** 2]                           # loss before any step is taken
    for _ in range(steps):                     # take `steps` descent steps
        w = w - eta * 2 * w                    # gradient step on f(w) = w squared
        curve.append(w ** 2)                   # store the loss after each step
    return curve                               # one loss value per step

plt.figure(figsize=(6, 3))                     # a wide, short figure
for eta, colour in [(0.05, "tab:blue"), (0.3, "tab:green"), (1.0, "tab:red")]:   # three rates
    plt.plot(losses(eta), marker="o", ms=3, color=colour, label=f"eta = {eta}")   # one curve each
plt.yscale("log")                              # the losses span decades, so use a log axis
plt.xlabel("descent step")                     # x-axis: how many updates have happened
plt.ylabel("loss f(w) = w^2")                  # y-axis: the value we are minimising
plt.title("eta = 0.3 is fastest here; eta = 1.0 stays at loss 4 forever")   # takeaway
plt.legend()                                   # name the three curves
plt.show()                                     # render the figure

**Line by line**

- `import matplotlib.pyplot as plt` — loads the plotting library.
- `import numpy as np` — NumPy is used by the interactive sections that follow.
- `def losses(eta, steps=12, start=2.0)` — one function returning a whole curve.
- `w = start` — reset the weight for each learning rate.
- `curve = [w ** 2]` — the loss at step 0, before any update.
- `for _ in range(steps)` — a fixed budget of 12 updates.
- `w = w - eta * 2 * w` — the update rule written compactly.
- `curve.append(w ** 2)` — append the loss at the new weight.
- `return curve` — hand the list back to the caller.
- `plt.figure(figsize=(6, 3))` — create a wide, short canvas.
- `for eta, colour in [...]` — loop over the three learning rates and their colours.
- `plt.plot(...)` — draw one loss curve per rate, with small dots at the steps.
- `plt.yscale("log")` — a log axis keeps all three curves visible.
- `plt.xlabel(...)` / `plt.ylabel(...)` — name the step count and the loss.
- `plt.title(...)` — state the comparison in one sentence.
- `plt.legend()` / `plt.show()` — label the curves, then draw them.

## 4. Interactive: choose your own learning rate

Drag the slider and the descent path redraws from the same start. The title reports the weight and loss after ten steps, so you can watch the number improve and then start bouncing as η passes 1.

In [ ]:
from ipywidgets import interact                # sliders that rerun a function
import matplotlib.pyplot as plt                # plotting for every frame

def show(eta=0.1):                             # eta comes from the slider
    steps = 10                                 # fixed budget of ten updates
    ws = [2.0]                                 # the path starts at w = 2
    for _ in range(steps):                     # walk downhill ten times
        ws.append(ws[-1] - eta * 2 * ws[-1])   # one gradient step from the last weight
    curve = [w ** 2 for w in ws]               # loss at every visited weight

    plt.figure(figsize=(6, 3))                 # a fresh figure for each drag
    plt.plot(range(steps + 1), curve, marker="o", ms=3, color="tab:purple")   # loss versus step
    plt.xlabel("descent step")                 # x-axis label
    plt.ylabel("loss f(w)")                    # y-axis label
    plt.title(f"eta = {eta:.2f}: last w = {ws[-1]:+.3f}, loss = {curve[-1]:.3f}")   # live readout
    plt.show()                                 # draw the frame

interact(show, eta=(0.01, 1.05, 0.01))         # attach the slider, step size 0.01

**Line by line**

- `from ipywidgets import interact` — Colab ships the widgets library.
- `def show(eta=0.1)` — the function that redraws one frame.
- `steps = 10` — the number of updates shown per frame.
- `ws = [2.0]` — the list of visited weights, starting at 2.
- `for _ in range(steps)` — one iteration per update.
- `ws.append(...)` — apply w ← w − η·2w and store the result.
- `curve = [w ** 2 for w in ws]` — compute the loss at each stored weight.
- `plt.figure(...)` — a new canvas per slider position.
- `plt.plot(...)` — join the loss values, dots at each step.
- `plt.xlabel(...)` / `plt.ylabel(...)` — label step and loss.
- `plt.title(...)` — live readout of the final weight and loss.
- `plt.show()` — render the frame.
- `interact(show, eta=(0.01, 1.05, 0.01))` — the slider, its range and step.

## 5. When it breaks: η = 1.0

At η = 1.0 the update multiplies the weight by 1 − 2η = −1. The sign flips every step, so the weight bounces between +2 and −2 and the loss stays at 4. The walk is not descending at all.

In [ ]:
import matplotlib.pyplot as plt                # plotting library
w = 2.0                                        # the same starting weight
eta = 1.0                                      # the step size that breaks the descent
ws = [w]                                       # record the path

for _ in range(6):                             # six steps are enough to see the pattern
    w = w - eta * 2 * w                        # 2 - 2*2 = -2, then -2 - 2*(-2) = 2
    ws.append(w)                               # store the new weight
    print(f"w = {w:+.1f}  loss = {w ** 2:.1f}")   # loss never leaves 4.0

plt.figure(figsize=(6, 3))                     # create the figure
plt.plot(range(len(ws)), ws, marker="o", color="tab:red")   # weight against step
plt.axhline(0, color="grey", linewidth=0.8)    # the target the descent should approach
plt.xlabel("descent step")                     # x-axis label
plt.ylabel("weight w")                         # y-axis label
plt.title("eta = 1.0: the weight flips +2 <-> -2 and never lands")   # takeaway
plt.show()                                     # render the figure

**Line by line**

- `import matplotlib.pyplot as plt` — plotting for the zigzag.
- `w = 2.0` — start from the usual weight.
- `eta = 1.0` — the critical step size, where 1 − 2η = −1.
- `ws = [w]` — the path begins at 2.
- `for _ in range(6)` — six updates, enough to show the loop.
- `w = w - eta * 2 * w` — each step negates the weight.
- `ws.append(w)` — record the flipped weight.
- `print(f"w = ...")` — show that the loss is always 4.0.
- `plt.figure(...)` — create the canvas.
- `plt.plot(...)` — draw weight against step as a zigzag.
- `plt.axhline(0, ...)` — mark the bottom of the bowl at w = 0.
- `plt.xlabel(...)` / `plt.ylabel(...)` / `plt.title(...)` — label the failure.
- `plt.show()` — render.

## 6. Compare: loss after eight steps, six rates

One number per learning rate makes the trade-off concrete. Rates below 0.5 leave something on the table, η = 0.5 lands exactly on zero, and rates above 0.5 are worse than doing nothing useful.

In [ ]:
import matplotlib.pyplot as plt                # plotting library
rates = [0.05, 0.1, 0.3, 0.5, 0.9, 1.0]       # six candidate learning rates
final = []                                     # one final loss per rate

for eta in rates:                              # try each rate from the same start
    w = 2.0                                    # reset the weight every time
    for _ in range(8):                         # eight descent steps
        w = w - eta * 2 * w                    # the same update rule
    final.append(w ** 2)                       # loss after the eighth step

plt.figure(figsize=(6, 3))                     # create the figure
plt.bar([str(r) for r in rates], final, color="tab:orange")   # one bar per rate
plt.xlabel("learning rate eta")                # x-axis label
plt.ylabel("loss after 8 steps")               # y-axis label
plt.title("eta = 0.5 hits zero in one step; eta = 1.0 is stuck at 4")   # takeaway
for i, v in enumerate(final):                  # annotate every bar
    plt.text(i, v, f"{v:.2f}", ha="center", va="bottom", fontsize=8)   # value above the bar
plt.show()                                     # render the figure

**Line by line**

- `import matplotlib.pyplot as plt` — plotting for the bar chart.
- `rates = [...]` — the six learning rates being compared.
- `final = []` — will hold one loss per rate.
- `for eta in rates` — one run per rate.
- `w = 2.0` — every run starts at the same weight.
- `for _ in range(8)` — eight updates per run.
- `w = w - eta * 2 * w` — the update rule again.
- `final.append(w ** 2)` — record the loss at the end of the run.
- `plt.figure(...)` / `plt.bar(...)` — one bar per learning rate.
- `plt.xlabel(...)` / `plt.ylabel(...)` — rate on x, final loss on y.
- `plt.title(...)` — the comparison in one sentence.
- `for i, v in enumerate(final)` — walk the bars with their index and height.
- `plt.text(...)` — write each bar's value on top of it.
- `plt.show()` — render.

## Your turn

Set the slider to η = 0.49 and then 0.51 and predict which ends lower after ten steps. Then explain why η = 0.5 reaches w = 0 after a single step while η = 0.51 overshoots to the other side.